In [1]:
# Cell 1: Setup
# ==============
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

# --- Paths ---
TRAIN_PATH   = '../../data/semeval2010/raw/TRAIN_FILE.TXT'
TEST_PATH    = '../../data/semeval2010/raw/TEST_FILE.TXT'
FULL_PATH    = '../../data/semeval2010/raw/TEST_FILE_FULL.TXT'
KEY_PATH     = '../../data/semeval2010/processed/answer_key.txt'

SCORER_PATH  = '../../scorer/semeval2010_task8_scorer-v1.2.pl'
CHECKER_PATH = '../../scorer/semeval2010_task8_format_checker.pl'

RESULTS_DIR      = '../../results/reproduction/semeval2010/'
PREDICTIONS_PATH = RESULTS_DIR + 'predictions_multiclass.txt'
SCORER_OUT_PATH  = RESULTS_DIR + 'scorer_output.txt'

# --- Imports ---
from src.shared.data_loader import load_semeval_train
from src.naive_bayes.data_loader import load_train_data, load_test_data, save_key_for_scorer
save_key_for_scorer(FULL_PATH, KEY_PATH)

from src.naive_bayes.features import create_vectorizer, extract_local_context
from src.naive_bayes.naive_bayes_model import build_naive_bayes_pipeline
from src.naive_bayes.evaluation import (
    save_predictions_official_format,
    run_format_checker,
    run_official_scorer,
)

print('Setup complete.')

Setup complete.


In [2]:
# Cell 2: Explore the Data / Show Feature Extraction
# ====================================================
from collections import Counter

train_examples = load_semeval_train(TRAIN_PATH, label_mode='full')

print("\n=== LABEL DISTRIBUTION IN TRAINING SET ===")
label_counts = Counter(ex['label'] for ex in train_examples)
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    pct = count / len(train_examples) * 100
    print(f"  {label:<35} {count:>5}  ({pct:.1f}%)")

print("\n=== SAMPLE TRAINING EXAMPLE ===")
for ex in train_examples[:1]:
    print(f"\n  ID:       {ex['id']}")
    print(f"  Sentence: {ex['sentence']}")
    print(f"  E1: '{ex['e1']}'   E2: '{ex['e2']}'")
    print(f"  Label:    {ex['label']}")

print("\n=== LOCAL CONTEXT FEATURE EXTRACTION (window=2) ===")
for ex in train_examples[:1]:
    context = extract_local_context(ex['sentence'], window=2)
    print(f"\n Full sentence:   {ex['sentence']}")
    print(f"Feature string:  '{context}'")
    print(f"Label:           {ex['label']}")
    print()


=== LABEL DISTRIBUTION IN TRAINING SET ===
  Other                                1410  (17.6%)
  Entity-Destination(e1,e2)             844  (10.5%)
  Cause-Effect(e2,e1)                   659  (8.2%)
  Member-Collection(e2,e1)              612  (7.6%)
  Entity-Origin(e1,e2)                  568  (7.1%)
  Message-Topic(e1,e2)                  490  (6.1%)
  Component-Whole(e2,e1)                471  (5.9%)
  Component-Whole(e1,e2)                470  (5.9%)
  Instrument-Agency(e2,e1)              407  (5.1%)
  Product-Producer(e2,e1)               394  (4.9%)
  Content-Container(e1,e2)              374  (4.7%)
  Cause-Effect(e1,e2)                   344  (4.3%)
  Product-Producer(e1,e2)               323  (4.0%)
  Content-Container(e2,e1)              166  (2.1%)
  Entity-Origin(e2,e1)                  148  (1.8%)
  Message-Topic(e2,e1)                  144  (1.8%)
  Instrument-Agency(e1,e2)               97  (1.2%)
  Member-Collection(e1,e2)               78  (1.0%)
  Entity-Destinati

In [3]:
# Cell 3: Train the Naive Bayes Model
# =====================================

# Load training data (already applies feature extraction)
train_features, train_labels, _ = load_train_data(TRAIN_PATH)

# Build the model
vectorizer = create_vectorizer()  # CountVectorizer, unigrams
model = build_naive_bayes_pipeline(vectorizer, alpha=0.8)

# Train
print("\nTraining...")
model.fit(train_features, train_labels)
print("Done!")

# How large is the vocabulary?
vocab_size = len(model.named_steps['bow'].vocabulary_)
print(f"Vocabulary size: {vocab_size} unique words or word combinations")


Training...
Done!
Vocabulary size: 48053 unique words or word combinations


In [4]:
# Cell 4: Generate Predictions
# =============================
from collections import Counter

# Load test sentences (no labels as we predict those)
test_features, test_examples = load_test_data(TEST_PATH)

print(f"Predicting for {len(test_examples)} test sentences...")

# Get predictions
predictions = model.predict(test_features)

# Extract IDs (needed for the output file)
test_ids = [ex['id'] for ex in test_examples]

# Show prediction distribution
print("\n=== PREDICTION DISTRIBUTION ===")
pred_counts = Counter(predictions)
for label, count in sorted(pred_counts.items(), key=lambda x: -x[1]):
    print(f"  {label:<35} {count:>5}")

Predicting for 2717 test sentences...

=== PREDICTION DISTRIBUTION ===
  Other                                1220
  Entity-Destination(e1,e2)             316
  Cause-Effect(e2,e1)                   222
  Entity-Origin(e1,e2)                  164
  Content-Container(e1,e2)              161
  Member-Collection(e2,e1)              151
  Component-Whole(e1,e2)                137
  Message-Topic(e1,e2)                  115
  Cause-Effect(e1,e2)                    71
  Component-Whole(e2,e1)                 52
  Instrument-Agency(e2,e1)               36
  Product-Producer(e2,e1)                28
  Content-Container(e2,e1)               21
  Product-Producer(e1,e2)                14
  Entity-Origin(e2,e1)                    7
  Instrument-Agency(e1,e2)                2


In [5]:
# Cell 5: Save and Validate Prediction Format
# ==================================

# Save in the EXACT format the scorer needs
save_predictions_official_format(test_ids, predictions, PREDICTIONS_PATH)

# ALWAYS run the format checker before scoring. If this shows errors the results will be wrong or rejected
format_ok = run_format_checker(PREDICTIONS_PATH, CHECKER_PATH)

Predictions saved to: ../../results/reproduction/semeval2010/predictions_multiclass.txt
Running official format checker...

<<< The file format is OK.

✓ Format is valid — ready to score!


In [6]:
# Cell 6: Run the Official Scorer
# ==================================
# The official score = macro-averaged F1 for (9+1)-way evaluation WITH directionality, excluding the "Other" class.

if format_ok:
    scorer_output = run_official_scorer(
        PREDICTIONS_PATH,
        KEY_PATH,
        SCORER_PATH,
        SCORER_OUT_PATH
    )
else:
    print("Fix format errors first before running the scorer!")

Running official semeval scorer
<<< (2*9+1)-WAY EVALUATION (USING DIRECTIONALITY)>>>:

Confusion matrix:
        C-E1 C-E2 C-W1 C-W2 C-C1 C-C2 E-D1 E-D2 E-O1 E-O2 I-A1 I-A2 M-C1 M-C2 M-T1 M-T2 P-P1 P-P2  _O_ <-- classified as
      +-----------------------------------------------------------------------------------------------+ -SUM- skip ACTUAL
 C-E1 |  69   30    1    0    0    0    2    0    2    0    0    0    0    0    0    0    0    0   30 |  134    0  134
 C-E2 |   1  154    0    0    0    0    0    0    8    0    0    0    0    0    1    0    0    0   30 |  194    0  194
 C-W1 |   0    3   93    1    2    0    2    0    0    0    0    0    0    2    1    0    0    0   58 |  162    0  162
 C-W2 |   0    0   16   44    1    0    4    0    2    0    0    0    0    4    2    0    0    0   77 |  150    0  150
 C-C1 |   0    0    0    0  118    0    5    0    0    0    0    0    0    0    0    0    0    0   30 |  153    0  153
 C-C2 |   0    0    0    2    7   20    1    0    0    0 